# Fine-Tuning Model Classifier NLP Penyakit ASEAN
Notebook ini digunakan untuk melatih ulang (*fine-tune*) model `xlm-roberta-base` / `indobert-base-p1` menggunakan dataset berlabel 31 Master Penyakit ASEAN.

In [ ]:
# 1. Install Library Dependencies
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas

In [ ]:
# 2. Load Dataset CSV
import pandas as pd
import numpy as np
import torch

# Upload dataset_master_combined_all.csv ke Colab atau sesuaikan path
df = pd.read_csv('dataset_master_combined_all.csv')
print(f"Total baris data: {len(df)}")
print(df['disease_label'].value_counts())
df.head()

In [ ]:
# 3. Preprocessing Data & Encode Labels
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Buat text gabungan judul + isi berita
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

# Encode String Labels to IDs
unique_labels = sorted(df['disease_label'].unique().tolist())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
df['label'] = df['disease_label'].map(label2id)

train_df, eval_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
print(f"Train set: {len(train_df)} samples, Eval set: {len(eval_df)} samples")

MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(texts):
    return tokenizer(texts, truncation=True, max_length=512, padding='max_length')

In [ ]:
# 4. Define PyTorch Dataset
class DiseaseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_encodings = tokenize_fn(train_df['text'].tolist())
eval_encodings = tokenize_fn(eval_df['text'].tolist())

train_dataset = DiseaseDataset(train_encodings, train_df['label'].tolist())
eval_dataset = DiseaseDataset(eval_encodings, eval_df['label'].tolist())

In [ ]:
# 5. Setup Model & Metrics (Macro F1 >= 0.80 Target)
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': macro_f1}

training_args = TrainingArguments(
    output_dir='./fine_tuned_output',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# 6. Start Fine-Tuning Training
trainer.train()

# 7. Save Fine-Tuned Model Output for System Deployment
output_dir = './fine-tuned'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model berhasil disimpan di {output_dir}. Siap di-deploy ke RunPod / Docker!")